[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/01_calibration_and_platt_scaling.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Calibration and Platt scaling

Allow about 40 minutes. Can a score be read as an AF probability?
You will build a reliability summary, fit a recalibration map on held-out
recordings, and see why better probabilities can change decisions without
changing the ranking. Recording labels are real CinC labels; every model score
is simulated.

The action is **send this recording for AF review**, priced as in the core
notebook: $C_{TP}$ = 1, $C_{FP}$ = 1, $C_{FN}$ = 5 and $C_{TN}$ = 0, in illustrative loss units.
Replace each `...` and run the checker below it; if you are stuck for about
10 minutes, use the stuck switch below the function, where there is one, or open the worked copy in `extension/solutions/`.
A later cell that needs an unfinished problem prints `Waiting for ...`: once that
problem is done or switched, click in the waiting cell, choose **Runtime > Run
before**, then run the waiting cell.

In [ ]:
import numpy as np
from IPython.display import display
from clinical_utility import data, reference, calibration, comparison, logistic
from clinical_utility import extension_checks as check, extension_plots as view

costs = dict(reference.DEFAULT_COSTS)
development = data.load_arrays("development")
calibration_set = data.load_arrays("calibration")
evaluation = data.load_arrays("evaluation")

## Read a reliability plot

A reliability plot groups recordings by score. Each point's horizontal
position is the mean score in a group; its vertical position is the fraction
of that group with AF. A point below the diagonal means the scores there are
too high. Equal-count (quantile) bins give the tails some data, so their widths
vary. Sort the scores and the labels together before splitting them into bins.

### The Brier score

The Brier score is the mean squared difference between the stated probability
and the outcome: one for AF and zero otherwise. A recording with AF and a score of
0.9 has squared error 0.01. Take 4 recordings with scores
0.9, 0.3, 0.7 and 0.1, of which only the first has AF. Their Brier
score is 0.15. In two equal-count bins, the lower bin has mean score
0.2 and AF fraction 0.0; the upper bin has mean score 0.8 and AF
fraction 0.5. Both bins sit below the diagonal: the scores are too high in each.

Average the squared errors over recordings, not over bins. The Brier score
also rewards separating AF from non-AF, so it is not a pure calibration
measure, and a small Brier score does not choose a clinical preference.

### Problem A: Make

Write `probability_summary(p, y, bins)`. Return a dictionary: arrays named
`predicted`, `observed` and `n`, one entry per bin, and a number `brier`. Use
`np.argsort` and `np.array_split` to form equal-count bins. The checker's first
case has unsorted scores, so it can tell when scores were sorted without their
labels.

In [ ]:
def probability_summary(p, y, bins=8):
    # order = ...; groups = np.array_split(order, min(bins, len(p)))
    return {"predicted": ..., "observed": ..., "n": ..., "brier": ...}
_ = check.run("cal_summary", probability_summary)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# probability_summary = check.use_reference("probability_summary")

In [ ]:
# Your summary of the three models on the evaluation split.
if check.ready(probability_summary=probability_summary):
    models = {"Random forest": "random_forest", "Raw ResNet": "resnet",
              "Recalibrated ResNet": "resnet_calibrated"}
    summaries = {name: probability_summary(evaluation[column], evaluation["af_present"])
                 for name, column in models.items()}
    display(view.reliability(summaries))
    for name, column in models.items():
        fit = calibration.calibration_intercept_slope(evaluation[column], evaluation["af_present"])
        print(f"{name:20}  Brier {summaries[name]['brier']:.3f}  "
              f"intercept {fit['intercept']:+.2f}  slope {fit['slope']:.2f}")

### Intercept and slope

These fits work on the log-odds scale. The logit of a probability `p` is its
log-odds, `logit(p) = log(p / (1 - p))`, and the sigmoid,
`sigmoid(z) = 1 / (1 + exp(-z))`, turns log-odds back into a probability. A
score of 0.8 has odds 4, so its logit is log 4 = 1.386, and the
sigmoid of 1.386 is 0.8 again.

Calibration-in-the-large fits only an intercept, with the score's log-odds as
a fixed offset: `logit(P(AF)) = a + logit(score)`. Its ideal is zero, and a
negative `a` means the scores are too high overall. The calibration slope comes
from a second fit, `logit(P(AF)) = a + b * logit(score)`, with both free. Its
ideal is one, and a slope below one means the log-odds are too extreme. Read
only the slope from that second fit: its intercept is not
calibration-in-the-large.

The raw ResNet fails both. In the top tenth of its evaluation scores the mean
score is 0.944, but only 49.8% of those recordings have AF. A few bins
and one Brier score do not establish calibration in every subgroup or in a new
population.

## Fit the map, then freeze it

Platt scaling maps a score to `sigmoid(a + b * logit(score))`. The intercept
`a` and slope `b` come from a logistic regression of the AF label on
`logit(score)`, fitted on held-out recordings.

### Problem B: Predict

Suppose the fitted slope `b` is positive. After recalibration, will the exact
minimum priced loss over all cutoffs be lower, higher or the same? Store
`"lower"`, `"higher"` or `"same"`, and write one sentence of reasoning in
`reason`.

In [ ]:
ranking_prediction = ...
reason = """
"""
_ = check.run("cal_predict", ranking_prediction)

A map with a positive slope keeps the order of the scores and their ties.
Every cutoff on the raw scores has an equivalent cutoff on the recalibrated
scores, so the set of achievable decisions, and its minimum loss, is
unchanged. What moves is the score that identifies each decision.

The fit maximises `sum(y*z - logaddexp(0, z))`, where `z = X @ beta` and `X`
has an intercept column and the feature `logit(score)`. Each Newton step uses:

- `p = sigmoid(X @ beta)`;
- the gradient `g = X.T @ (y - p)`;
- the information `H = X.T @ ((p*(1-p))[:, None] * X)`;
- the update `beta += solve(H, g)`.

For one AF recording, its contribution to the intercept's gradient is `1 - p`;
for a recording without AF it is `-p`. The sum pulls the fitted probabilities
towards the observed number with AF.

### Problem C: Modify

Complete `my_logit`, then the gradient and information lines in `fit_platt`.
The inputs lie strictly between zero and one. Use `logit(score)` as the feature:
regressing on the probability itself is a legitimate calibration model, but it
does not implement the logit-linear specification used here. The checker fits
the calibration split, so its answer matches the supplied `resnet_calibrated`
column. `fit_platt` returns `{"intercept": a, "slope": b}`.

In [ ]:
def my_logit(p):
    """The log-odds, element by element."""
    return ...


def fit_platt(scores, y):
    X = np.column_stack([np.ones(len(y)), my_logit(scores)])
    beta = np.zeros(2)
    for _ in range(100):
        p = logistic.sigmoid(X @ beta)
        gradient = ...
        information = ...
        step = np.linalg.solve(information, gradient)
        beta += step
        if np.max(np.abs(step)) < 1e-10:
            break
    return {"intercept": float(beta[0]), "slope": float(beta[1])}
_ = check.run("cal_fit", fit_platt)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# fit_platt = check.use_reference("fit_platt")

In [ ]:
# Fit on calibration only, then apply the frozen map to evaluation.
if check.ready(fit_platt=fit_platt):
    fitted = fit_platt(calibration_set["resnet"], calibration_set["af_present"])
    recalibrated = calibration.platt_apply(evaluation["resnet"], fitted)
    print("Calibration-only fit:", fitted)
    print("Largest difference from the supplied column:",
          np.max(np.abs(recalibrated - evaluation["resnet_calibrated"])))

Platt scaling is not the only map. Isotonic regression fits a non-decreasing
step function by pool-adjacent-violators (PAV): sort by score, start from the
observed AF fractions, and merge neighbouring blocks whose fractions decrease
into their size-weighted mean. Its flat steps create ties, so it can remove
achievable thresholds, and its flexibility needs more calibration data.

In [ ]:
isotonic = calibration.isotonic_fit(calibration_set["resnet"], calibration_set["af_present"])
isotonic_scores = calibration.isotonic_apply(evaluation["resnet"], isotonic)
print("Isotonic evaluation Brier:", calibration.brier_score(isotonic_scores, evaluation["af_present"]))
if check.ready(probability_summary=probability_summary):
    display(view.reliability({"Isotonic": probability_summary(isotonic_scores,
                                                              evaluation["af_present"])}))

### Problem D: Fix the bug

A colleague reports how well the Platt map works. The code runs and the Brier
score looks good. Fix `calibration_brier` in place so that it reports an
honest figure, then say in `leak` what the original version leaked.

In [ ]:
def calibration_brier(splits):
    """Evaluation Brier score of the Platt-recalibrated ResNet."""
    evaluation = splits["evaluation"]
    fitted = calibration.platt_fit(evaluation["resnet"], evaluation["af_present"])
    recalibrated = calibration.platt_apply(evaluation["resnet"], fitted)
    return calibration.brier_score(recalibrated, evaluation["af_present"])


leak = """
"""
splits = {"development": development, "calibration": calibration_set, "evaluation": evaluation}
_ = check.run("cal_split", calibration_brier)

## Transfer the prior, not the ranking

On development, raw and recalibrated ResNet scores reach the same exact minimum
loss, 922 and 922. Acting at a stated preference is different. At
preferences 0.1, 0.2 and 0.4, recalibration raises net benefit by
0.0054, 0.0102 and 0.0321 per recording. These preferences are
alternatives; only the middle one follows the default matrix.

Under prior shift, the score distribution within each reference class stays
fixed while the prevalence changes. Bayes' rule multiplies the odds by the new
prior odds divided by the old prior odds. On the log-odds scale that is an
intercept shift:
`logit(p_new) = logit(p_old) + logit(prevalence_new) - logit(prevalence_old)`.

### Problem E: Make

Implement the shift in `shift_prior`. Below, the evaluation recordings are
resampled to a prevalence of 2%. Compare the mean predicted
probability with the observed AF fraction, and fit nothing on the resampled
outcomes. The old prevalence is the calibration split's observed AF fraction,
not the mean score in the new population.

In [ ]:
def shift_prior(p, old_prevalence, new_prevalence):
    return ...
_ = check.run("cal_shift", shift_prior)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# shift_prior = check.use_reference("shift_prior")

In [ ]:
y = evaluation["af_present"]
negative, positive = np.flatnonzero(~y), np.flatnonzero(y)
target = .02
keep = int(round(len(negative) * target / (1 - target)))
chosen = np.random.default_rng(20261106).choice(positive, keep, replace=False)
rows = np.r_[negative, np.sort(chosen)]
if check.ready(shift_prior=shift_prior):
    p = evaluation["resnet_calibrated"][rows]
    transferred = shift_prior(p, calibration_set["af_present"].mean(), target)
    print("Mean probability before and after the shift, and observed AF fraction:",
          p.mean(), transferred.mean(), y[rows].mean())

# Achievable minimum loss, then net benefit at three stated preferences.
for model in ("resnet", "resnet_calibrated"):
    sweep = reference.sweep_thresholds(data.load_records(model, "development"), costs)
    print(model, "development minimum loss:", reference.best_thresholds(sweep)[0]["total_loss"])
for preference in (.1, .2, .4):
    print(preference, {m: comparison.policy_net_benefit(development[m], development["af_present"],
                                                        preference)
                       for m in ("resnet", "resnet_calibrated")})

The resampled mean moves from 0.0699 to 0.0201; the observed prevalence is
0.0202. That agreement checks this prior-shift scenario only. A new device,
more noise or a different mix of N and O rhythms can change the score
distribution within each class, and then an intercept-only repair is not
enough.

## Before you finish

- Explain the difference between a Brier score and a reliability plot.
- Say which recordings fitted the map and which assessed it.
- Explain how the same ranking supports different decisions at a stated preference.
- State the assumption behind the prior shift.

**Discuss:** If the calibration slope also changes in a new service, what
evidence would you need before trusting an intercept-only update?